# MAE encoder model training

# 1. Load frozen monocyte HVG input

## Dependencies, paths, and training parameters

In [ ]:
from pathlib import Path
import gc
import math
import os
import random
import time
from collections import defaultdict
from itertools import combinations

import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
from sklearn.neighbors import NearestNeighbors
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset, DataLoader, Sampler

# Documented private input and output path variables.
# Private AnnData objects and metadata workbooks are not distributed.
analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path

PROJECT_ROOT = resolve_analysis_path(
    os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai"))
)
HVG_ADATA_PATH = resolve_analysis_path(
    os.environ.get("MAE_HVG_H5AD", os.path.join("outputs", "ai", "adata_mono_MAE_HVG5000.h5ad"))
)
METADATA_PATH = resolve_analysis_path(
    os.environ.get("MAE_METADATA_XLSX", "Metadata.xlsx")
)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

DISEASE_COL = "disease"
PATIENT_COL = "patient_id"
SAMPLE_COL = "sample_id"
DISEASES = ["CAR-T_CRS", "COVID19", "SLE"]

BATCH_SIZE = 256
MASK_RATE = 0.20
MASK_TOKEN = -1.0
NUM_WORKERS = 0
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
MAX_GRAD_NORM = 1.0
N_EPOCHS = 30
FINAL_SEEDS = [20260810, 20260811, 20260812, 20260813, 20260814]

HIDDEN_DIM_1 = 512
HIDDEN_DIM_2 = 128
LATENT_DIM = 64
ENCODE_BATCH_SIZE = 1024

QC_CELLS_PER_DISEASE = 2000
QC_K = 30
QC_SEED = 20260810


## Derived training state

In [ ]:
adata_m = sc.read_h5ad(HVG_ADATA_PATH)

if PATIENT_COL not in adata_m.obs.columns:
    metadata = pd.read_excel(METADATA_PATH)
    sample_to_patient = dict(
        zip(
            metadata[SAMPLE_COL].astype(str),
            metadata[PATIENT_COL].astype(str),
        )
    )
    adata_m.obs[PATIENT_COL] = (
        adata_m.obs[SAMPLE_COL]
        .astype(str)
        .map(sample_to_patient)
    )

if SAMPLE_COL not in adata_m.obs.columns:
    raise ValueError(f"Missing required sample column: {SAMPLE_COL}")

if not sp.isspmatrix_csr(adata_m.X):
    adata_m.X = sp.csr_matrix(adata_m.X)

INPUT_DIM = adata_m.n_vars
STEPS_PER_EPOCH = math.ceil(adata_m.n_obs / BATCH_SIZE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

hierarchy = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

for idx, row in enumerate(
    adata_m.obs[[DISEASE_COL, PATIENT_COL, SAMPLE_COL]].itertuples(index=False)
):
    disease = str(row[0])
    patient = str(row[1])
    sample = str(row[2])
    hierarchy[disease][patient][sample].append(idx)

print("Device:", DEVICE)
print("Input dimension:", INPUT_DIM)
print("Steps per epoch:", STEPS_PER_EPOCH)


# 2. Helper functions

In [ ]:
class HierarchicalBalancedBatchSampler(Sampler):

    def __init__(
        self,
        hierarchy,
        batch_size,
        steps_per_epoch,
        seed=20260810
    ):
        """
        Parameters
        ----------
        hierarchy
            Nested dictionary:
            disease -> patient -> sample -> cell indices

        batch_size
            Number of cells drawn for one optimization step.

        steps_per_epoch
            Number of minibatches generated per epoch.
            Because sampling is with replacement, an "epoch"
            is defined by optimization steps rather than one
            exhaustive pass through every cell.

        seed
            Base random seed for reproducible sampling.
        """

        self.hierarchy = hierarchy
        self.diseases = list(hierarchy.keys())

        self.batch_size = int(batch_size)
        self.steps_per_epoch = int(steps_per_epoch)

        self.seed = int(seed)
        self.epoch = 0

        if len(self.diseases) < 2:
            raise ValueError(
                "At least two diseases are required."
            )

        if self.batch_size < len(self.diseases):
            raise ValueError(
                "batch_size must be >= number of diseases."
            )


    def set_epoch(self, epoch):
        """
        Change random sampling deterministically between epochs.
        """
        self.epoch = int(epoch)


    def __len__(self):
        return self.steps_per_epoch


    def __iter__(self):

        # New but reproducible sequence for each epoch
        rng = np.random.default_rng(
            self.seed + self.epoch
        )

        n_diseases = len(self.diseases)

        for _ in range(self.steps_per_epoch):

            # ------------------------------------------------
            # Allocate approximately equal cells per disease
            # within EACH batch
            # ------------------------------------------------

            n_base = (
                self.batch_size // n_diseases
            )

            remainder = (
                self.batch_size % n_diseases
            )

            disease_counts = {
                disease: n_base
                for disease in self.diseases
            }

            # If batch size is not divisible by 3,
            # distribute remaining slots randomly so that
            # no disease systematically receives the extra cell.
            if remainder > 0:

                extra_diseases = rng.choice(
                    self.diseases,
                    size=remainder,
                    replace=False
                )

                for disease in extra_diseases:
                    disease_counts[disease] += 1


            batch_indices = []

            # ------------------------------------------------
            # disease -> patient -> sample -> cell
            # ------------------------------------------------

            for disease in self.diseases:

                patients = list(
                    self.hierarchy[disease].keys()
                )

                n_draws = disease_counts[disease]

                for _ in range(n_draws):

                    # 1. Patient uniformly within disease
                    patient = rng.choice(
                        patients
                    )

                    samples = list(
                        self.hierarchy[disease][patient].keys()
                    )

                    # 2. Sample uniformly within patient
                    sample = rng.choice(
                        samples
                    )

                    cells = self.hierarchy[
                        disease
                    ][patient][sample]

                    # 3. Cell uniformly within sample
                    cell_idx = rng.choice(
                        cells
                    )

                    batch_indices.append(
                        int(cell_idx)
                    )


            # Randomize order inside the batch so cells are
            # not grouped by disease when passed to the model.
            rng.shuffle(batch_indices)

            yield batch_indices

In [ ]:
# dataset that returns cell indices only 
class CellIndexDataset(Dataset):

    def __init__(self, n_cells):
        self.n_cells = int(n_cells)

    def __len__(self):
        return self.n_cells

    def __getitem__(self, idx):
        return int(idx)


index_dataset = CellIndexDataset(
    adata_m.n_obs
)

In [ ]:
def make_masked_collate_fn(
    X,
    mask_rate=0.20,
    mask_token=-1.0
):

    def collate_fn(cell_indices):

        # --------------------------------------------
        # 1. Convert indices to NumPy
        # --------------------------------------------

        idx = np.asarray(
            cell_indices,
            dtype=np.int64
        )


        # --------------------------------------------
        # 2. Extract ONE minibatch from X
        # --------------------------------------------

        if sp.issparse(X):

            x_np = (
                X[idx, :]
                .toarray()
                .astype(
                    np.float32,
                    copy=False
                )
            )

        else:

            x_np = np.asarray(
                X[idx, :],
                dtype=np.float32
            )


        # --------------------------------------------
        # 3. Convert to PyTorch tensor
        # --------------------------------------------

        x = torch.from_numpy(x_np)


        # --------------------------------------------
        # 4. Identify detected genes
        #
        # Only positive log1p values are eligible
        # for masking.
        # --------------------------------------------

        detected = x > 0


        # --------------------------------------------
        # 5. Randomly mask ~20% of detected genes
        #
        # torch.rand_like gives an independent
        # U[0,1) random value for every matrix entry.
        # --------------------------------------------

        mask = (
            torch.rand_like(x) < mask_rate
        ) & detected


        # --------------------------------------------
        # 6. Safety:
        # ensure every cell with detected genes has
        # at least one masked target.
        # --------------------------------------------

        n_detected = detected.sum(dim=1)
        n_masked = mask.sum(dim=1)

        need_one_mask = (
            (n_detected > 0)
            & (n_masked == 0)
        )

        rows_need_mask = torch.where(
            need_one_mask
        )[0]

        for row in rows_need_mask:

            detected_genes = torch.where(
                detected[row]
            )[0]

            chosen_position = torch.randint(
                low=0,
                high=detected_genes.numel(),
                size=(1,)
            ).item()

            chosen_gene = detected_genes[
                chosen_position
            ]

            mask[row, chosen_gene] = True


        # --------------------------------------------
        # 7. Produce corrupted model input
        # --------------------------------------------

        x_masked = x.clone()

        x_masked[mask] = mask_token


        # --------------------------------------------
        # Return:
        #
        # x
        #   uncorrupted reconstruction target
        #
        # x_masked
        #   encoder input
        #
        # mask
        #   exactly which positions contribute to
        #   reconstruction loss
        #
        # cell_idx
        #   lets us trace every cell back to AnnData
        # --------------------------------------------

        return {
            "x": x,
            "x_masked": x_masked,
            "mask": mask,
            "cell_idx": torch.from_numpy(idx)
        }


    return collate_fn

In [ ]:
class MaskedExpressionAutoencoder(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim_1=512,
        hidden_dim_2=128,
        latent_dim=64
    ):
        super().__init__()

        # ----------------------------------------------------
        # Encoder
        # 5000 -> 512 -> 128 -> 64
        # ----------------------------------------------------

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim_1
            ),

            nn.LayerNorm(
                hidden_dim_1
            ),

            nn.GELU(),

            nn.Linear(
                hidden_dim_1,
                hidden_dim_2
            ),

            nn.LayerNorm(
                hidden_dim_2
            ),

            nn.GELU(),

            nn.Linear(
                hidden_dim_2,
                latent_dim
            )
        )


        # ----------------------------------------------------
        # Decoder
        # 64 -> 128 -> 512 -> 5000
        # ----------------------------------------------------

        self.decoder = nn.Sequential(

            nn.Linear(
                latent_dim,
                hidden_dim_2
            ),

            nn.LayerNorm(
                hidden_dim_2
            ),

            nn.GELU(),

            nn.Linear(
                hidden_dim_2,
                hidden_dim_1
            ),

            nn.LayerNorm(
                hidden_dim_1
            ),

            nn.GELU(),

            nn.Linear(
                hidden_dim_1,
                input_dim
            )
        )


    def encode(self, x):
        """
        Return latent representation only.

        IMPORTANT:
        After training, clean/unmasked expression will be
        passed through this function to obtain the final
        cell representation.
        """
        return self.encoder(x)


    def decode(self, z):
        return self.decoder(z)


    def forward(self, x):

        z = self.encode(x)
        reconstruction = self.decode(z)

        return reconstruction, z

In [ ]:
# Masked reconstruction loss

def masked_mse_per_cell(
    prediction,
    target,
    mask
):
    """
    MSE is:

    1. calculated only for intentionally masked genes;
    2. averaged within each cell;
    3. then averaged across cells.

    Thus every sampled cell receives approximately equal
    influence irrespective of its number of detected genes.
    """

    squared_error = (
        prediction - target
    ).pow(2)

    mask_float = mask.float()

    # Sum masked errors separately for each cell
    error_sum_per_cell = (
        squared_error
        * mask_float
    ).sum(dim=1)

    # Number of masked genes in each cell
    n_masked_per_cell = (
        mask_float
        .sum(dim=1)
        .clamp_min(1.0)
    )

    # Mean masked error per cell
    loss_per_cell = (
        error_sum_per_cell
        /
        n_masked_per_cell
    )

    # Every cell contributes one value
    return loss_per_cell.mean()

In [ ]:
def set_training_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
def build_train_loader(seed):

    batch_sampler = HierarchicalBalancedBatchSampler(
        hierarchy=hierarchy,
        batch_size=BATCH_SIZE,
        steps_per_epoch=STEPS_PER_EPOCH,
        seed=seed
    )

    collate_fn = make_masked_collate_fn(
        X=adata_m.X,
        mask_rate=MASK_RATE,
        mask_token=MASK_TOKEN
    )

    loader = DataLoader(
        dataset=index_dataset,
        batch_sampler=batch_sampler,
        collate_fn=collate_fn,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    return loader

In [ ]:
def train_one_seed(
    seed,
    n_epochs=N_EPOCHS
):

    set_training_seed(seed)

    # --------------------------------------------------------
    # Fresh loader
    # --------------------------------------------------------

    train_loader_seed = build_train_loader(
        seed=seed
    )


    # --------------------------------------------------------
    # Fresh model initialization
    # --------------------------------------------------------

    model = MaskedExpressionAutoencoder(
        input_dim=INPUT_DIM,
        hidden_dim_1=HIDDEN_DIM_1,
        hidden_dim_2=HIDDEN_DIM_2,
        latent_dim=LATENT_DIM
    ).to(DEVICE)


    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )


    history = []


    # ========================================================
    # Epoch loop
    # ========================================================

    for epoch in range(n_epochs):

        start_time = time.time()

        # Makes hierarchical sampling reproducibly different
        # from one epoch to another.
        train_loader_seed.batch_sampler.set_epoch(
            epoch
        )

        model.train()

        epoch_loss = 0.0
        n_batches = 0


        # ====================================================
        # Minibatch loop
        # ====================================================

        for batch in train_loader_seed:

            x = batch["x"].to(
                DEVICE,
                non_blocking=True
            )

            x_masked = batch["x_masked"].to(
                DEVICE,
                non_blocking=True
            )

            mask = batch["mask"].to(
                DEVICE,
                non_blocking=True
            )


            # -----------------------------------------------
            # Forward
            # -----------------------------------------------

            optimizer.zero_grad(
                set_to_none=True
            )

            reconstruction, z = model(
                x_masked
            )


            # -----------------------------------------------
            # Masked-only reconstruction loss
            # -----------------------------------------------

            loss = masked_mse_per_cell(
                prediction=reconstruction,
                target=x,
                mask=mask
            )


            # -----------------------------------------------
            # Backpropagation
            # -----------------------------------------------

            loss.backward()


            # -----------------------------------------------
            # Gradient clipping
            # -----------------------------------------------

            clip_grad_norm_(
                model.parameters(),
                max_norm=MAX_GRAD_NORM
            )


            optimizer.step()


            epoch_loss += loss.item()
            n_batches += 1


        # ====================================================
        # Epoch summary
        # ====================================================

        mean_epoch_loss = (
            epoch_loss / n_batches
        )

        elapsed = (
            time.time() - start_time
        )

        history.append({
            "seed": seed,
            "epoch": epoch + 1,
            "loss": mean_epoch_loss,
            "seconds": elapsed
        })


        print(
            f"Seed {seed} | "
            f"Epoch {epoch + 1:02d}/{n_epochs} | "
            f"Loss = {mean_epoch_loss:.5f} | "
            f"{elapsed:.1f} s"
        )


    history_df = pd.DataFrame(
        history
    )

    return model, history_df

In [ ]:
# save checkpoints with CPU tensors to avoid GPU memory issues when loading later

def get_cpu_state_dict(model):
    """
    Copy model parameters to CPU before checkpointing.
    """

    return {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    }

In [ ]:

def encode_all_cells(
    model,
    X,
    batch_size=1024
):
    """
    Encode every cell using the COMPLETE, unmasked
    5000-gene expression profile.

    Returns
    -------
    Z : np.ndarray
        Shape = n_cells × latent_dim
        dtype = float32
    """

    model.eval()

    n_cells = X.shape[0]

    Z = np.empty(
        (n_cells, LATENT_DIM),
        dtype=np.float32
    )


    with torch.inference_mode():

        for start in range(
            0,
            n_cells,
            batch_size
        ):

            end = min(
                start + batch_size,
                n_cells
            )


            # -----------------------------------------------
            # Extract sparse rows and densify ONLY this batch
            # -----------------------------------------------

            if sp.issparse(X):

                x_np = (
                    X[start:end, :]
                    .toarray()
                    .astype(
                        np.float32,
                        copy=False
                    )
                )

            else:

                x_np = np.asarray(
                    X[start:end, :],
                    dtype=np.float32
                )


            x_tensor = (
                torch.from_numpy(x_np)
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            # -----------------------------------------------
            # Encoder only
            # -----------------------------------------------

            z = model.encode(
                x_tensor
            )


            Z[start:end, :] = (
                z.cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=False
                )
            )


    return Z


In [ ]:
# helper for loading each frozen model
def load_frozen_model(
    checkpoint_path
):

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )


    model = MaskedExpressionAutoencoder(
        input_dim=checkpoint["input_dim"],
        hidden_dim_1=checkpoint["hidden_dim_1"],
        hidden_dim_2=checkpoint["hidden_dim_2"],
        latent_dim=checkpoint["latent_dim"]
    )


    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model = model.to(
        DEVICE
    )

    model.eval()

    return model, checkpoint

# 3. Training-input integrity checks

In [ ]:
# inspect resulted structure
for disease in DISEASES:

    n_patients = len(
        hierarchy[disease]
    )

    n_samples = sum(
        len(samples)
        for samples in hierarchy[disease].values()
    )

    n_cells = sum(
        len(cell_indices)
        for samples in hierarchy[disease].values()
        for cell_indices in samples.values()
    )

    print(
        f"{disease}: "
        f"{n_patients} patients | "
        f"{n_samples} samples | "
        f"{n_cells:,} cells"
    )

In [ ]:
# ============================================================
# Batch sampler
# ============================================================

batch_sampler = HierarchicalBalancedBatchSampler(
    hierarchy=hierarchy,
    batch_size=BATCH_SIZE,
    steps_per_epoch=STEPS_PER_EPOCH,
    seed=SEED
)


# ============================================================
# Collate / masking function
# ============================================================

masked_collate_fn = make_masked_collate_fn(
    X=adata_m.X,
    mask_rate=MASK_RATE,
    mask_token=MASK_TOKEN
)


# ============================================================
# DataLoader
# ============================================================

train_loader = DataLoader(
    dataset=index_dataset,
    batch_sampler=batch_sampler,
    collate_fn=masked_collate_fn,
    num_workers=NUM_WORKERS
)

In [ ]:
batch = next(
    iter(train_loader)
)

print("Original expression:")
print(batch["x"].shape)
print(batch["x"].dtype)

print("\nMasked expression:")
print(batch["x_masked"].shape)

print("\nMask:")
print(batch["mask"].shape)
print(batch["mask"].dtype)

print("\nCell indices:")
print(batch["cell_idx"].shape)

In [ ]:
x = batch["x"]
mask = batch["mask"]

detected = x > 0

actual_mask_rate = (
    mask.sum().item()
    /
    detected.sum().item()
)

print(
    f"Actual masked fraction among detected genes: "
    f"{actual_mask_rate:.3f}"
)

In [ ]:
assert torch.all(
    batch["x"][batch["mask"]] > 0
)

assert torch.all(
    batch["x_masked"][batch["mask"]]
    == MASK_TOKEN
)

print("Masking integrity check passed.")

In [ ]:
# verify disease balancing
cell_idx = (
    batch["cell_idx"]
    .numpy()
)

batch_meta = adata_m.obs.iloc[
    cell_idx
]

print(
    batch_meta[DISEASE_COL]
    .value_counts()
)

In [ ]:
# check whether patients are also balanced uniformly within each disease
N_QC_BATCHES = 200

qc_records = []

sampler_iter = iter(batch_sampler)

for batch_number in range(N_QC_BATCHES):

    indices = next(sampler_iter)

    tmp = adata_m.obs.iloc[
        indices
    ][
        [
            DISEASE_COL,
            PATIENT_COL,
            SAMPLE_COL
        ]
    ].copy()

    tmp["qc_batch"] = batch_number

    qc_records.append(tmp)


qc_df = pd.concat(
    qc_records,
    axis=0,
    ignore_index=True
)

In [ ]:
print("Disease sampling proportions:\n")

print(
    qc_df[DISEASE_COL]
    .value_counts(normalize=True)
    .sort_index()
)

In [ ]:
# patient level sampling distribution
patient_draws = (
    qc_df
    .groupby(
        [
            DISEASE_COL,
            PATIENT_COL
        ],
        observed=True
    )
    .size()
    .rename("n_draws")
    .reset_index()
)

for disease in DISEASES:

    tmp = patient_draws[
        patient_draws[DISEASE_COL]
        == disease
    ]

    print(f"\n{disease}")

    print(
        tmp["n_draws"]
        .describe()
    )

    cv = (
        tmp["n_draws"].std()
        /
        tmp["n_draws"].mean()
    )

    print(
        f"Patient-draw CV: {cv:.3f}"
    )

# 4. Five-seed masked-autoencoder training and latent export

In [ ]:
# ============================================================
# Final fixed seed set
# ============================================================


FINAL_N_EPOCHS = N_EPOCHS

MODEL_DIR = Path("Fig6_MAE_models")
MODEL_DIR.mkdir(exist_ok=True)

HISTORY_DIR = Path("Fig6_MAE_history")
HISTORY_DIR.mkdir(exist_ok=True)


In [ ]:
# run remaining seeds

for seed in FINAL_SEEDS:

    checkpoint_path = (
        MODEL_DIR /
        f"Fig6_MAE_seed_{seed}.pt"
    )

    history_path = (
        HISTORY_DIR /
        f"Fig6_MAE_seed_{seed}_history.csv"
    )

    # --------------------------------------------------------
    # Pilot seed has already been trained.
    #
    # If you already saved it using the previous filename
    # outside MODEL_DIR, either copy that checkpoint here
    # or skip this seed manually.
    # --------------------------------------------------------

    if checkpoint_path.exists():

        print(
            f"Seed {seed}: checkpoint already exists, skipping."
        )
        continue


    print(
        f"\n{'=' * 60}\n"
        f"Training final seed {seed}\n"
        f"{'=' * 60}"
    )


    model_seed, history_seed = train_one_seed(
        seed=seed,
        n_epochs=FINAL_N_EPOCHS
    )


    checkpoint = {

        "seed": seed,

        "input_dim": INPUT_DIM,
        "hidden_dim_1": HIDDEN_DIM_1,
        "hidden_dim_2": HIDDEN_DIM_2,
        "latent_dim": LATENT_DIM,

        "mask_rate": MASK_RATE,
        "mask_token": MASK_TOKEN,

        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,

        "batch_size": BATCH_SIZE,
        "steps_per_epoch": STEPS_PER_EPOCH,

        "n_epochs": FINAL_N_EPOCHS,

        "model_state_dict":
            get_cpu_state_dict(model_seed)
    }


    torch.save(
        checkpoint,
        checkpoint_path
    )

    history_seed.to_csv(
        history_path,
        index=False
    )


    # --------------------------------------------------------
    # Free GPU memory before next seed
    # --------------------------------------------------------

    del model_seed
    del history_seed
    del checkpoint

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
for seed in FINAL_SEEDS:

    path = (
        MODEL_DIR /
        f"Fig6_MAE_seed_{seed}.pt"
    )

    print(
        seed,
        "OK" if path.exists() else "MISSING"
    )

In [ ]:
LATENT_DIR = Path(
    "Fig6_MAE_latent"
)

LATENT_DIR.mkdir(
    exist_ok=True
)


for seed in FINAL_SEEDS:

    print(
        f"\nEncoding seed {seed}..."
    )


    checkpoint_path = (
        MODEL_DIR /
        f"Fig6_MAE_seed_{seed}.pt"
    )


    model_seed, checkpoint = (
        load_frozen_model(
            checkpoint_path
        )
    )


    Z_seed = encode_all_cells(
        model=model_seed,
        X=adata_m.X,
        batch_size=ENCODE_BATCH_SIZE
    )

    # QC
    assert Z_seed.shape == (
        adata_m.n_obs,
        LATENT_DIM
    )

    assert np.isfinite(
        Z_seed
    ).all()


    print(
        "Shape:",
        Z_seed.shape
    )

    print(
        "Mean:",
        Z_seed.mean()
    )

    print(
        "SD:",
        Z_seed.std()
    )

    # Save
    np.save(
        LATENT_DIR /
        f"Fig6_Z_seed_{seed}.npy",
        Z_seed
    )

    # Free memory
    del Z_seed
    del model_seed
    del checkpoint

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
adata_m.uns["Fig6_MAE"] = {

    "status":
        "frozen",

    "n_features":
        int(adata_m.n_vars),

    "latent_dim":
        LATENT_DIM,

    "architecture":
        [5000, 512, 128, 64, 128, 512, 5000],

    "mask_rate":
        MASK_RATE,

    "mask_strategy":
        "detected_genes_only",

    "mask_token":
        MASK_TOKEN,

    "sampling":
        "equal disease -> equal patient -> equal sample -> cell",

    "batch_size":
        BATCH_SIZE,

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "epochs":
        FINAL_N_EPOCHS,

    "seeds":
        FINAL_SEEDS,

    "clinical_metadata_used_for_training":
        False
}

In [ ]:
print("=== FINAL SAVE CHECK ===")

for seed in FINAL_SEEDS:

    model_path = (
        MODEL_DIR /
        f"Fig6_MAE_seed_{seed}.pt"
    )

    latent_path = (
        LATENT_DIR /
        f"Fig6_Z_seed_{seed}.npy"
    )

    history_path = (
        HISTORY_DIR /
        f"Fig6_MAE_seed_{seed}_history.csv"
    )

    print(
        f"\nSeed {seed}\n"
        f"  model:   {model_path.exists()} "
        f"{model_path.stat().st_size / 1e6:.1f} MB"
        if model_path.exists()
        else f"\nSeed {seed}\n  model:   MISSING"
    )

    print(
        f"  latent:  {latent_path.exists()} "
        f"{latent_path.stat().st_size / 1e6:.1f} MB"
        if latent_path.exists()
        else "  latent:  MISSING"
    )

    print(
        f"  history: {history_path.exists()}"
    )


In [ ]:
# Verify saved files
for seed in FINAL_SEEDS:

    model_path = (
        MODEL_DIR /
        f"Fig6_MAE_seed_{seed}.pt"
    )

    latent_path = (
        LATENT_DIR /
        f"Fig6_Z_seed_{seed}.npy"
    )

    history_path = (
        HISTORY_DIR /
        f"Fig6_MAE_seed_{seed}_history.csv"
    )

    print(
        f"Seed {seed}: "
        f"model={model_path.exists()}, "
        f"latent={latent_path.exists()}, "
        f"history={history_path.exists()}"
    )

In [ ]:
# Load the five latent representations
Z_by_seed = {}

for seed in FINAL_SEEDS:

    latent_path = (
        LATENT_DIR /
        f"Fig6_Z_seed_{seed}.npy"
    )

    Z_by_seed[seed] = np.load(
        latent_path,
        mmap_mode="r"
    )

    print(
        f"Seed {seed}: "
        f"shape={Z_by_seed[seed].shape}, "
        f"dtype={Z_by_seed[seed].dtype}"
    )

In [ ]:
# Integrity check with adata_m
for seed in FINAL_SEEDS:

    Z = Z_by_seed[seed]

    assert Z.shape[0] == adata_m.n_obs, (
        f"Seed {seed}: cell-number mismatch "
        f"({Z.shape[0]} vs {adata_m.n_obs})"
    )

    assert Z.shape[1] == 64, (
        f"Seed {seed}: expected 64 latent dimensions, "
        f"found {Z.shape[1]}"
    )

print("All five latent representations match adata_m.")

# 5. Seed-neighborhood stability QC

In [ ]:
# Fixed balanced subset for seed-stability QC

rng = np.random.default_rng(QC_SEED)

qc_indices = []

for disease in DISEASES:
    disease_idx = np.where(
        adata_m.obs[DISEASE_COL]
        .astype(str)
        .to_numpy()
        == str(disease)
    )[0]

    n_take = min(
        len(disease_idx),
        QC_CELLS_PER_DISEASE,
    )

    chosen = rng.choice(
        disease_idx,
        size=n_take,
        replace=False,
    )

    qc_indices.append(chosen)

qc_indices = np.concatenate(qc_indices)

print(f"QC cells: {len(qc_indices):,}")


In [ ]:
# Seed-specific neighbors
# kNN for each seed
seed_neighbors = {}


for seed in FINAL_SEEDS:

    Z = np.load(
        LATENT_DIR /
        f"Fig6_Z_seed_{seed}.npy",
        mmap_mode="r"
    )


    Z_qc = np.asarray(
        Z[qc_indices, :],
        dtype=np.float32
    )


    nn_model = NearestNeighbors(
        n_neighbors=QC_K,
        metric="euclidean",
        algorithm="auto",
        n_jobs=-1
    )


    nn_model.fit(
        Z_qc
    )


    neighbors = (
        nn_model
        .kneighbors(
            return_distance=False
        )
    )


    seed_neighbors[seed] = (
        neighbors
    )

In [ ]:
# Pairwise neighborhood overlap

stability_records = []


for seed_a, seed_b in combinations(
    FINAL_SEEDS,
    2
):

    neigh_a = seed_neighbors[
        seed_a
    ]

    neigh_b = seed_neighbors[
        seed_b
    ]


    overlap_fraction = []


    for i in range(
        len(qc_indices)
    ):

        set_a = set(
            neigh_a[i]
        )

        set_b = set(
            neigh_b[i]
        )


        overlap = len(
            set_a.intersection(
                set_b
            )
        )


        overlap_fraction.append(
            overlap / QC_K
        )


    overlap_fraction = np.asarray(
        overlap_fraction
    )


    stability_records.append({

        "seed_a":
            seed_a,

        "seed_b":
            seed_b,

        "mean_knn_overlap":
            overlap_fraction.mean(),

        "median_knn_overlap":
            np.median(
                overlap_fraction
            ),

        "q25":
            np.quantile(
                overlap_fraction,
                0.25
            ),

        "q75":
            np.quantile(
                overlap_fraction,
                0.75
            )
    })


stability_df = pd.DataFrame(
    stability_records
)


stability_df

In [ ]:
stability_df.to_csv(
    "Fig6_MAE_seed_neighborhood_stability.csv",
    index=False
)